# Workshop: Building Gemma 3 from Scratch
## Notebook 3: Grouped Query Attention (GQA)

**Estimated Time: 15 minutes**

Standard Multi-Head Attention (MHA) gives every Query head its own Key and Value head. This is memory-intensive for large models. **Grouped Query Attention (GQA)** optimizes this by sharing one K and V head among multiple Query heads.

## Learning Objectives:
1. Contrast MHA, MQA (Multi-Query), and GQA.
2. Understand the efficiency gains of GQA.
3. Implement the tensor repetition logic used in GQA.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

batch_size = 1
seq_len = 8
num_heads = 8
num_kv_groups = 2
head_dim = 96  # Gemma 3: 768 / 8 = 96

group_size = num_heads // num_kv_groups  # = 4 for Gemma 3 (8/2)
print(f"Each KV head will be shared by {group_size} Query heads.")

Python version: 3.12.13 (main, Mar 10 2026, 18:15:41) [Clang 21.1.4 ]
PyTorch version: 2.5.1
Each KV head will be shared by 4 Query heads.


## 1. MHA vs MQA vs GQA

| Architecture   | Q heads | K heads | V heads | KV-Cache size |
|----------------|---------|---------|---------|---------------|
| **MHA**        | 8       | 8       | 8       | 8x            |
| **MQA**        | 8       | 1       | 1       | 1x            |
| **GQA** (Gemma 3) | 8    | 2       | 2       | 2x            |

Gemma 3 uses **GQA with 2 KV groups** -- a highly optimized sweet spot between MHA's quality and MQA's speed.

## 2. Generating Q, K, V with Groups

In [2]:
# Shape: (batch, num_heads/groups, seq_len, head_dim)
Q = torch.randn(batch_size, num_heads, seq_len, head_dim)
K = torch.randn(batch_size, num_kv_groups, seq_len, head_dim)
V = torch.randn(batch_size, num_kv_groups, seq_len, head_dim)

print(f"Q heads: {Q.shape[1]}")
print(f"K heads: {K.shape[1]}")
print(f"V heads: {V.shape[1]}")
print(f"KV-Cache saved: {(1 - num_kv_groups/num_heads) * 100:.0f}% vs MHA")

Q heads: 8
K heads: 2
V heads: 2
KV-Cache saved: 75% vs MHA


## 3. The Repetition Trick

In [3]:
# Expand K and V to match Q's number of heads
K_expanded = K.repeat_interleave(group_size, dim=1)
V_expanded = V.repeat_interleave(group_size, dim=1)

print(f"Q heads:     {Q.shape[1]}")
print(f"Expanded K:  {K_expanded.shape[1]}")
print(f"Expanded V:  {V_expanded.shape[1]}")
assert K_expanded.shape[1] == Q.shape[1]
print('OK: K and V successfully expanded to match Q!')

Q heads:     8
Expanded K:  8
Expanded V:  8
OK: K and V successfully expanded to match Q!


## 4. GQA Attention with QK-Norm (Gemma 3)

In [4]:
# QK-Norm attention with GQA
Q_norm = F.normalize(Q, p=2, dim=-1)
K_norm = F.normalize(K_expanded, p=2, dim=-1)

scores = torch.matmul(Q_norm, K_norm.transpose(-2, -1)) / math.sqrt(head_dim)
attention_weights = F.softmax(scores, dim=-1)

output = torch.matmul(attention_weights, V_expanded)
print(f"Attention output shape: {output.shape}")

Attention output shape: torch.Size([1, 8, 8, 96])


## 5. Why GQA?

In inference, we store K and V in a **KV Cache**. By using fewer KV heads, we drastically reduce the memory footprint of the cache, allowing for larger batch sizes and longer contexts.

For Gemma 3 with 2 KV groups:
- KV-cache memory = 2/8 = 25% of full MHA
- Combined with 5:1 local/global layers: total KV reduction < 15%

## Exercise:
Implement the `SimpleGQA` class. It should project an input $x$ to $Q, K, V$, expand $K$ and $V$, and compute the attention output.

In [5]:
class SimpleGQA(nn.Module):
    def __init__(self, d_in, n_heads, n_kv_groups, h_dim):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups
        self.h_dim = h_dim
        self.group_size = n_heads // n_kv_groups
        
        # GQA projects to fewer K/V dimensions
        self.W_q = nn.Linear(d_in, n_heads * h_dim, bias=False)
        self.W_k = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.W_v = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * h_dim, d_in, bias=False)
        
    def forward(self, x):
        B, T, C = x.shape
        
        # 1. Project
        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        
        # 2. Expand K, V groups to match Q heads
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
        
        # 3. QK-Norm attention (Gemma 3)
        q_norm = F.normalize(q, p=2, dim=-1)
        k_norm = F.normalize(k, p=2, dim=-1)
        scores = torch.matmul(q_norm, k_norm.transpose(-2, -1)) / math.sqrt(self.h_dim)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)
        
        # 4. Reshape and project out
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)

# Test your implementation
d_in, n_h, n_kv, h_d = 96, 8, 2, 96
model = SimpleGQA(d_in, n_h, n_kv, h_d)
x = torch.randn(1, 5, d_in)
output = model(x)
print(f"OK: Success! GQA Output shape: {output.shape}")
assert output.shape == x.shape

OK: Success! GQA Output shape: torch.Size([1, 5, 96])



**Solution:**

```python
class SimpleGQA(nn.Module):
    def __init__(self, d_in, n_heads, n_kv_groups, h_dim):
        super().__init__()
        self.n_heads, self.n_kv_groups, self.h_dim = n_heads, n_kv_groups, h_dim
        self.group_size = n_heads // n_kv_groups
        self.W_q = nn.Linear(d_in, n_heads * h_dim, bias=False)
        self.W_k = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.W_v = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * h_dim, d_in, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
        q_norm = F.normalize(q, p=2, dim=-1)
        k_norm = F.normalize(k, p=2, dim=-1)
        scores = torch.matmul(q_norm, k_norm.transpose(-2, -1)) / math.sqrt(self.h_dim)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)
```
